In [4]:
import os
import re
import joblib
import json

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.cross_decomposition import PLSRegression
from sklearn.preprocessing import LabelBinarizer
from sklearn.base import BaseEstimator, ClassifierMixin
import numpy as np

from scipy.signal import savgol_filter

from sklearn.base import BaseEstimator, ClassifierMixin, TransformerMixin
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import StratifiedKFold, cross_val_predict, cross_val_score
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC

try:
    import tensorflow as tf
    from tensorflow.keras import Sequential
    from tensorflow.keras.layers import Conv1D, Dense, Dropout, Flatten, MaxPooling1D, InputLayer
    from tensorflow.keras.callbacks import EarlyStopping
    TENSORFLOW_AVAILABLE = True
except ImportError:
    TENSORFLOW_AVAILABLE = False
from sklearn.cross_decomposition import PLSRegression


# =========================================================
# TRANSFORMADORES DE PRÉ-PROCESSAMENTO
# =========================================================

class SavgolFilter(BaseEstimator, TransformerMixin):

    def __init__(self, window_length=11, polyorder=2, deriv=0):
        self.window_length = window_length
        self.polyorder = polyorder
        self.deriv = deriv

    def fit(self, X, y=None):
        return self

    def transform(self, X):
        return savgol_filter(
            X,
            window_length=self.window_length,
            polyorder=self.polyorder,
            deriv=self.deriv,
            axis=1,
        )


class SNV(BaseEstimator, TransformerMixin):

    def fit(self, X, y=None):
        return self

    def transform(self, X):
        mean = np.mean(X, axis=1, keepdims=True)
        std = np.std(X, axis=1, ddof=1, keepdims=True)
        std[std == 0] = 1
        return (X - mean) / std


# =========================================================
# MODELOS CUSTOMIZADOS
# =========================================================

class PLSDAClassifier(BaseEstimator, ClassifierMixin):
    """PLS-DA binário usando PLSRegression e limiar em 0.5."""

    def __init__(self, n_components=2):
        self.n_components = n_components

    def fit(self, X, y):
        self.label_encoder_ = LabelEncoder()
        y_encoded = self.label_encoder_.fit_transform(y)
        if len(self.label_encoder_.classes_) != 2:
            raise ValueError('PLS-DA nesta implementação espera apenas duas classes.')

        self.pls_ = PLSRegression(n_components=self.n_components)
        self.pls_.fit(X, y_encoded)
        return self

    def predict(self, X):
        scores = self.pls_.predict(X).ravel()
        predicted = (scores >= 0.5).astype(int)
        return self.label_encoder_.inverse_transform(predicted)
    


class PLSDAMulticlass(BaseEstimator, ClassifierMixin):

    def __init__(self, n_components=10):
        self.n_components = n_components

    def fit(self, X, y):

        self.lb_ = LabelBinarizer()
        Y = self.lb_.fit_transform(y)

        if Y.ndim == 1:
            Y = Y.reshape(-1, 1)

        self.classes_ = self.lb_.classes_

        self.models_ = []

        for i in range(Y.shape[1]):
            pls = PLSRegression(
                n_components=min(
                    self.n_components,
                    X.shape[0] - 1,
                    X.shape[1]
                )
            )

            pls.fit(X, Y[:, i])
            self.models_.append(pls)

        return self

    def predict(self, X):

        scores = np.column_stack([
            model.predict(X).ravel()
            for model in self.models_
        ])

        idx = np.argmax(scores, axis=1)

        return self.classes_[idx]

    def predict_proba(self, X):

        scores = np.column_stack([
            model.predict(X).ravel()
            for model in self.models_
        ])

        scores = np.maximum(scores, 0)

        row_sum = scores.sum(axis=1, keepdims=True)
        row_sum[row_sum == 0] = 1

        return scores / row_sum
 
    
def extrair_numero(label):
    match = re.search(r'\d+', label)
    return int(match.group()) if match else None


def make_savgol(window_length=11, polyorder=2):
    return ('savgol', SavgolFilter(window_length=window_length, polyorder=polyorder))


def make_snv():
    return ('snv', SNV())


def make_scaler():
    return ('scaler', StandardScaler())

pipeline com todos os modelos fazendo classificacao binaria com k-fold

In [ ]:
import os
import re
import joblib
import json

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from scipy.signal import savgol_filter

from sklearn.base import BaseEstimator, ClassifierMixin, TransformerMixin
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import StratifiedKFold, cross_val_predict, cross_val_score
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC

try:
    import tensorflow as tf
    from tensorflow.keras import Sequential
    from tensorflow.keras.layers import Conv1D, Dense, Dropout, Flatten, MaxPooling1D, InputLayer
    from tensorflow.keras.callbacks import EarlyStopping
    TENSORFLOW_AVAILABLE = True
except ImportError:
    TENSORFLOW_AVAILABLE = False
from sklearn.cross_decomposition import PLSRegression


# =========================================================
# TRANSFORMADORES DE PRÉ-PROCESSAMENTO
# =========================================================

class SavgolFilter(BaseEstimator, TransformerMixin):

    def __init__(self, window_length=11, polyorder=2, deriv=0):
        self.window_length = window_length
        self.polyorder = polyorder
        self.deriv = deriv

    def fit(self, X, y=None):
        return self

    def transform(self, X):
        return savgol_filter(
            X,
            window_length=self.window_length,
            polyorder=self.polyorder,
            deriv=self.deriv,
            axis=1,
        )


class SNV(BaseEstimator, TransformerMixin):

    def fit(self, X, y=None):
        return self

    def transform(self, X):
        mean = np.mean(X, axis=1, keepdims=True)
        std = np.std(X, axis=1, ddof=1, keepdims=True)
        std[std == 0] = 1
        return (X - mean) / std


# =========================================================
# MODELOS CUSTOMIZADOS
# =========================================================

class PLSDAClassifier(BaseEstimator, ClassifierMixin):
    """PLS-DA binário usando PLSRegression e limiar em 0.5."""

    def __init__(self, n_components=2):
        self.n_components = n_components

    def fit(self, X, y):
        self.label_encoder_ = LabelEncoder()
        y_encoded = self.label_encoder_.fit_transform(y)
        if len(self.label_encoder_.classes_) != 2:
            raise ValueError('PLS-DA nesta implementação espera apenas duas classes.')

        self.pls_ = PLSRegression(n_components=self.n_components)
        self.pls_.fit(X, y_encoded)
        return self

    def predict(self, X):
        scores = self.pls_.predict(X).ravel()
        predicted = (scores >= 0.5).astype(int)
        return self.label_encoder_.inverse_transform(predicted)


class CNN1DClassifier(BaseEstimator, ClassifierMixin):
    """Wrapper scikit-learn para 1D-CNN com TensorFlow/Keras."""

    def __init__(self, epochs=30, batch_size=16, validation_split=0.2, verbose=0):
        self.epochs = epochs
        self.batch_size = batch_size
        self.validation_split = validation_split
        self.verbose = verbose

    def _build_model(self, input_length):
        model = Sequential([
            InputLayer(shape=(input_length, 1)),
            Conv1D(16, kernel_size=5, activation='relu', padding='same'),
            MaxPooling1D(pool_size=2),
            Conv1D(32, kernel_size=3, activation='relu', padding='same'),
            MaxPooling1D(pool_size=2),
            Flatten(),
            Dense(64, activation='relu'),
            Dropout(0.3),
            Dense(1, activation='sigmoid'),
        ])
        model.compile(
            optimizer='adam',
            loss='binary_crossentropy',
            metrics=['accuracy'],
        )
        return model

    def fit(self, X, y):
        if not TENSORFLOW_AVAILABLE:
            raise ImportError('TensorFlow não está instalado. Instale tensorflow para usar o modelo 1D-CNN.')

        self.label_encoder_ = LabelEncoder()
        y_encoded = self.label_encoder_.fit_transform(y).astype(np.float32)
        if len(self.label_encoder_.classes_) != 2:
            raise ValueError('O modelo 1D-CNN nesta implementação espera apenas duas classes.')

        X_cnn = X.astype(np.float32)[..., np.newaxis]
        self.model_ = self._build_model(X_cnn.shape[1])
        callback = EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)
        self.model_.fit(
            X_cnn,
            y_encoded,
            epochs=self.epochs,
            batch_size=self.batch_size,
            validation_split=self.validation_split,
            verbose=self.verbose,
            callbacks=[callback],
        )
        return self

    def predict(self, X):
        X_cnn = X.astype(np.float32)[..., np.newaxis]
        probabilities = self.model_.predict(X_cnn, verbose=0).ravel()
        predicted = (probabilities >= 0.5).astype(int)
        return self.label_encoder_.inverse_transform(predicted)


# =========================================================
# FUNÇÕES AUXILIARES
# =========================================================

def extrair_numero(label):
    match = re.search(r'\d+', label)
    return int(match.group()) if match else None


def make_savgol(window_length=11, polyorder=2):
    return ('savgol', SavgolFilter(window_length=window_length, polyorder=polyorder))


def make_snv():
    return ('snv', SNV())


def make_scaler():
    return ('scaler', StandardScaler())


def save_pipeline_summary_json(output_path, payload):
    with open(output_path, 'w', encoding='utf-8') as json_file:
        json.dump(payload, json_file, ensure_ascii=False, indent=2)


# =========================================================
# ESCOLHA DO LASER
# =========================================================

#laser = int(input('Qual laser utilizado? 532 ou 1064\n'))
laser = 1064

if laser not in [532, 1064]:
    raise ValueError('O valor deve ser 532 ou 1064')


# =========================================================
# LEITURA DO DATASET
# =========================================================

print('\nLendo dataset...')
file_path = f'dataset/laser{laser}.dat'

df = pd.read_csv(file_path, sep=';', header=None)

y_raw = df.iloc[0, 1:].values.astype(str)
X_full = df.iloc[1:, 1:].values.astype(float).T


# =========================================================
# SEPARAÇÃO DAS CLASSES
# =========================================================

X = []
y = []

for xi, yi in zip(X_full, y_raw):
    num = extrair_numero(yi)
    if laser == 532:
        if yi.startswith('A') and num is not None and num <= 8:
            X.append(xi)
            y.append('A')
        elif yi.startswith('R') and num is not None and num <= 8:
            X.append(xi)
            y.append('R')
    if laser == 1064:
        if yi.startswith('A') and num is not None and num <= 12:
            X.append(xi)
            y.append('A')
        elif yi.startswith('R') and num is not None and num <= 9:
            X.append(xi)
            y.append('R')

X = np.array(X)
y = np.array(y)

print(f'Formato do dataset: {X.shape}')
print(f'Labels: {np.unique(y)}')


# =========================================================
# CONFIGURAÇÃO DOS EXPERIMENTOS
# =========================================================

preprocessamentos = {
    'none': [],
    'savgol': [make_savgol()],
    'snv': [make_snv()],
    'scaler': [make_scaler()],
    'savgol_snv': [make_savgol(), make_snv()],
    'savgol_snv_scaler': [make_savgol(), make_snv(), make_scaler()],
}

modelos = {
    'svm_linear': SVC(kernel='linear', gamma='scale'),
    'svm_rbf': SVC(kernel='rbf', gamma='scale'),
    'svm_poly': SVC(kernel='poly', gamma='scale'),
    'svm_sigmoid': SVC(kernel='sigmoid', gamma='scale'),
    'random_forest': RandomForestClassifier(n_estimators=200, random_state=42),
    'pls_da': PLSDAClassifier(n_components=2),
}

if TENSORFLOW_AVAILABLE:
    modelos['cnn_1d'] = CNN1DClassifier(epochs=30, batch_size=16, validation_split=0.2, verbose=0)
else:
    print('TensorFlow não encontrado: o modelo 1D-CNN será listado, mas não poderá ser treinado até instalar tensorflow.')
    modelos['cnn_1d'] = CNN1DClassifier(epochs=30, batch_size=16, validation_split=0.2, verbose=0)


# =========================================================
# K-FOLD CROSS VALIDATION
# =========================================================

kfold = StratifiedKFold(
    n_splits=8,
    shuffle=True,
    random_state=42,
)


# =========================================================
# PASTAS DE SAÍDA
# =========================================================

os.makedirs(f'models_intra_classe/laser{laser}', exist_ok=True)
os.makedirs(f'plots_intra_classe/laser{laser}', exist_ok=True)


# =========================================================
# TREINAMENTO E AVALIAÇÃO
# =========================================================

results = {}
pipeline_summaries = []
print('Executando Cross Validation...')

for prep_name, prep_steps in preprocessamentos.items():
    for model_name, model in modelos.items():
        pipeline_name = f'{prep_name}_{model_name}'
        print(f'Pipeline: {pipeline_name}')

        pipe = Pipeline([
            *prep_steps,
            ('model', model),
        ])

        try:
            scores = cross_val_score(
                pipe,
                X,
                y,
                cv=kfold,
                scoring='accuracy',
            )
        except Exception as exc:
            print(f'Falha ao validar {pipeline_name}: {exc}')
            continue

        mean_acc = scores.mean()
        std_acc = scores.std()
        results[pipeline_name] = mean_acc

        print(f'Acurácias: {scores}')
        print(f'Média: {mean_acc:.4f}')
        print(f'Desvio padrão: {std_acc:.4f}')

        try:
            pipe.fit(X, y)

            if model_name == 'cnn_1d' and TENSORFLOW_AVAILABLE:
                model_path = f'models/laser{laser}/{pipeline_name}.keras'
                pipe.named_steps['model'].model_.save(model_path)
            else:
                model_path = f'models/laser{laser}/{pipeline_name}.joblib'
                joblib.dump(pipe, model_path)

            print(f'Modelo salvo em: {model_path}')
        except Exception as exc:
            print(f'Não foi possível salvar {pipeline_name}: {exc}')

        try:
            y_pred = cross_val_predict(pipe, X, y, cv=kfold)
            cm = confusion_matrix(y, y_pred)
            disp = ConfusionMatrixDisplay(
                confusion_matrix=cm,
                display_labels=['Arabica', 'Robusta'],
            )

            fig, ax = plt.subplots(figsize=(5, 5))
            disp.plot(ax=ax)
            plt.title(f'Matriz de Confusão - {pipeline_name}')

            plot_path = f'plots/laser{laser}/{pipeline_name}_cm.png'
            plt.savefig(plot_path, dpi=300, bbox_inches='tight')
            plt.close()
            print(f'Plot salvo em: {plot_path}')
        except Exception as exc:
            print(f'Não foi possível gerar matriz de confusão para {pipeline_name}: {exc}')

        pipeline_summaries.append({
            'pipeline': pipeline_name,
            'labels': [str(label) for label in np.unique(y)],
            'n_splits': kfold.get_n_splits(),
            'accuracy_scores': [float(score) for score in scores],
            'mean_accuracy': float(mean_acc),
            'std_accuracy': float(std_acc),
        })


consolidated_summary_path = f'plots/laser{laser}/all_pipelines_summary.json'
save_pipeline_summary_json(consolidated_summary_path, {
    'experiment': 'binary_kfold',
    'laser': laser,
    'labels': [str(label) for label in np.unique(y)],
    'n_splits': kfold.get_n_splits(),
    'pipelines': pipeline_summaries,
})
print(f'Resumo consolidado salvo em: {consolidated_summary_path}')

# =========================================================
# RANKING FINAL
# =========================================================

print('==============================')
print('RANKING FINAL')
print('==============================')

for k, v in sorted(results.items(), key=lambda item: item[1], reverse=True):
    print(f'{k}: {v:.4f}')


# =========================================================
# PLOT FINAL
# =========================================================

if results:
    names = list(results.keys())
    values = list(results.values())

    plt.figure(figsize=(14, 6))
    bars = plt.bar(names, values)
    plt.ylabel('Accuracy')
    plt.xlabel('Pipeline')
    plt.title(f'Comparação dos Pipelines - Laser {laser}')
    plt.ylim(0, 1)
    plt.xticks(rotation=45, ha='right')

    for bar in bars:
        height = bar.get_height()
        plt.text(
            bar.get_x() + bar.get_width() / 2,
            height + 0.01,
            f'{height:.3f}',
            ha='center',
        )

    plt.tight_layout()
    final_plot = f'plots/laser{laser}/ranking_final.png'
    plt.savefig(final_plot, dpi=300)
    plt.close()
    print(f'Plot final salvo em: {final_plot}')

print('Processo concluído!')


Lendo dataset...


C:\Users\Pedro\AppData\Local\Temp\ipykernel_18544\1204704881.py:204: DtypeWarning: Columns (0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20,21,22,23,24,25,26,27,28,29,30,31,32,33,34,35,36,37,38,39,40,41,42,43,44,45,46,47,48,49,50,51,52,53,54,55,56,57,58,59,60,61,62,63,64,65,66,67,68,69,70,71,72,73,74,75,76,77,78,79,80,81,82,83,84,85,86,87,88,89,90,91,92,93,94,95,96,97,98,99,100,101,102,103,104,105,106,107,108,109,110,111,112,113,114,115,116,117,118,119,120,121,122,123,124,125,126,127,128,129,130,131,132,133,134,135,136,137,138,139,140,141,142,143,144,145,146,147,148,149,150,151,152,153,154,155,156,157,158,159,160,161,162,163,164,165,166,167,168,169,170,171,172,173,174,175,176,177,178,179,180,181,182,183,184,185,186,187,188,189,190,191,192,193,194,195,196,197,198,199,200,201,202,203,204,205,206,207,208,209,210,211,212,213,214,215,216,217,218,219,220,221,222,223,224,225,226,227,228,229,230,231,232,233,234,235,236,237,238,239,240,241,242,243,244,245,246,247,248,249,250,251,252,253,25

Formato do dataset: (550, 30262)
Labels: ['A' 'R']
Executando Cross Validation...
Pipeline: none_svm_linear
Acurácias: [1. 1. 1. 1. 1. 1. 1. 1.]
Média: 1.0000
Desvio padrão: 0.0000
Modelo salvo em: models/laser1064/none_svm_linear.joblib
Plot salvo em: plots/laser1064/none_svm_linear_cm.png
Resumo salvo em: plots/laser1064/none_svm_linear_summary.json
Pipeline: none_svm_rbf
Acurácias: [1.         1.         0.98550725 1.         1.         1.
 1.         1.        ]
Média: 0.9982
Desvio padrão: 0.0048
Modelo salvo em: models/laser1064/none_svm_rbf.joblib
Plot salvo em: plots/laser1064/none_svm_rbf_cm.png
Resumo salvo em: plots/laser1064/none_svm_rbf_summary.json
Pipeline: none_svm_poly
Acurácias: [1.         1.         0.98550725 0.98550725 1.         1.
 1.         0.98529412]
Média: 0.9945
Desvio padrão: 0.0071
Modelo salvo em: models/laser1064/none_svm_poly.joblib
Plot salvo em: plots/laser1064/none_svm_poly_cm.png
Resumo salvo em: plots/laser1064/none_svm_poly_summary.json
Pipeline

In [ ]:
import os
import re
import joblib
import json

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from scipy.signal import savgol_filter

from sklearn.base import BaseEstimator, ClassifierMixin, TransformerMixin
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import StratifiedKFold, cross_val_predict, cross_val_score
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC

try:
    import tensorflow as tf
    from tensorflow.keras import Sequential
    from tensorflow.keras.layers import Conv1D, Dense, Dropout, Flatten, MaxPooling1D, InputLayer
    from tensorflow.keras.callbacks import EarlyStopping
    TENSORFLOW_AVAILABLE = True
except ImportError:
    TENSORFLOW_AVAILABLE = False
from sklearn.cross_decomposition import PLSRegression


# =========================================================
# TRANSFORMADORES DE PRÉ-PROCESSAMENTO
# =========================================================

class SavgolFilter(BaseEstimator, TransformerMixin):

    def __init__(self, window_length=11, polyorder=2, deriv=0):
        self.window_length = window_length
        self.polyorder = polyorder
        self.deriv = deriv

    def fit(self, X, y=None):
        return self

    def transform(self, X):
        return savgol_filter(
            X,
            window_length=self.window_length,
            polyorder=self.polyorder,
            deriv=self.deriv,
            axis=1,
        )


class SNV(BaseEstimator, TransformerMixin):

    def fit(self, X, y=None):
        return self

    def transform(self, X):
        mean = np.mean(X, axis=1, keepdims=True)
        std = np.std(X, axis=1, ddof=1, keepdims=True)
        std[std == 0] = 1
        return (X - mean) / std


# =========================================================
# MODELOS CUSTOMIZADOS
# =========================================================

class PLSDAClassifier(BaseEstimator, ClassifierMixin):
    """PLS-DA binário usando PLSRegression e limiar em 0.5."""

    def __init__(self, n_components=2):
        self.n_components = n_components

    def fit(self, X, y):
        self.label_encoder_ = LabelEncoder()
        y_encoded = self.label_encoder_.fit_transform(y)
        if len(self.label_encoder_.classes_) != 2:
            raise ValueError('PLS-DA nesta implementação espera apenas duas classes.')

        self.pls_ = PLSRegression(n_components=self.n_components)
        self.pls_.fit(X, y_encoded)
        return self

    def predict(self, X):
        scores = self.pls_.predict(X).ravel()
        predicted = (scores >= 0.5).astype(int)
        return self.label_encoder_.inverse_transform(predicted)


class CNN1DClassifier(BaseEstimator, ClassifierMixin):
    """Wrapper scikit-learn para 1D-CNN com TensorFlow/Keras."""

    def __init__(self, epochs=30, batch_size=16, validation_split=0.2, verbose=0):
        self.epochs = epochs
        self.batch_size = batch_size
        self.validation_split = validation_split
        self.verbose = verbose

    def _build_model(self, input_length):
        model = Sequential([
            InputLayer(shape=(input_length, 1)),
            Conv1D(16, kernel_size=5, activation='relu', padding='same'),
            MaxPooling1D(pool_size=2),
            Conv1D(32, kernel_size=3, activation='relu', padding='same'),
            MaxPooling1D(pool_size=2),
            Flatten(),
            Dense(64, activation='relu'),
            Dropout(0.3),
            Dense(1, activation='sigmoid'),
        ])
        model.compile(
            optimizer='adam',
            loss='binary_crossentropy',
            metrics=['accuracy'],
        )
        return model

    def fit(self, X, y):
        if not TENSORFLOW_AVAILABLE:
            raise ImportError('TensorFlow não está instalado. Instale tensorflow para usar o modelo 1D-CNN.')

        self.label_encoder_ = LabelEncoder()
        y_encoded = self.label_encoder_.fit_transform(y).astype(np.float32)
        if len(self.label_encoder_.classes_) != 2:
            raise ValueError('O modelo 1D-CNN nesta implementação espera apenas duas classes.')

        X_cnn = X.astype(np.float32)[..., np.newaxis]
        self.model_ = self._build_model(X_cnn.shape[1])
        callback = EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)
        self.model_.fit(
            X_cnn,
            y_encoded,
            epochs=self.epochs,
            batch_size=self.batch_size,
            validation_split=self.validation_split,
            verbose=self.verbose,
            callbacks=[callback],
        )
        return self

    def predict(self, X):
        X_cnn = X.astype(np.float32)[..., np.newaxis]
        probabilities = self.model_.predict(X_cnn, verbose=0).ravel()
        predicted = (probabilities >= 0.5).astype(int)
        return self.label_encoder_.inverse_transform(predicted)


# =========================================================
# FUNÇÕES AUXILIARES
# =========================================================

def extrair_numero(label):
    match = re.search(r'\d+', label)
    return int(match.group()) if match else None


def make_savgol(window_length=11, polyorder=2):
    return ('savgol', SavgolFilter(window_length=window_length, polyorder=polyorder))


def make_snv():
    return ('snv', SNV())


def make_scaler():
    return ('scaler', StandardScaler())


def save_pipeline_summary_json(output_path, payload):
    with open(output_path, 'w', encoding='utf-8') as json_file:
        json.dump(payload, json_file, ensure_ascii=False, indent=2)


# =========================================================
# ESCOLHA DO LASER
# =========================================================

#laser = int(input('Qual laser utilizado? 532 ou 1064\n'))
laser = 532

if laser not in [532, 1064]:
    raise ValueError('O valor deve ser 532 ou 1064')


# =========================================================
# LEITURA DO DATASET
# =========================================================

print('\nLendo dataset...')
file_path = f'dataset/laser{laser}.dat'

df = pd.read_csv(file_path, sep=';', header=None)

y_raw = df.iloc[0, 1:].values.astype(str)
X_full = df.iloc[1:, 1:].values.astype(float).T


# =========================================================
# SEPARAÇÃO DAS CLASSES
# =========================================================

X = []
y = []

for xi, yi in zip(X_full, y_raw):
    num = extrair_numero(yi)
    if laser == 532:
        if yi.startswith('A') and num is not None and num <= 8:
            X.append(xi)
            y.append('A')
        elif yi.startswith('R') and num is not None and num <= 8:
            X.append(xi)
            y.append('R')
    if laser == 1064:
        if yi.startswith('A') and num is not None and num <= 12:
            X.append(xi)
            y.append('A')
        elif yi.startswith('R') and num is not None and num <= 9:
            X.append(xi)
            y.append('R')

X = np.array(X)
y = np.array(y)

print(f'Formato do dataset: {X.shape}')
print(f'Labels: {np.unique(y)}')


# =========================================================
# CONFIGURAÇÃO DOS EXPERIMENTOS
# =========================================================

preprocessamentos = {
    'none': [],
    'savgol': [make_savgol()],
    'snv': [make_snv()],
    'scaler': [make_scaler()],
    'savgol_snv': [make_savgol(), make_snv()],
    'savgol_snv_scaler': [make_savgol(), make_snv(), make_scaler()],
}

modelos = {
    'svm_linear': SVC(kernel='linear', gamma='scale'),
    'svm_rbf': SVC(kernel='rbf', gamma='scale'),
    'svm_poly': SVC(kernel='poly', gamma='scale'),
    'svm_sigmoid': SVC(kernel='sigmoid', gamma='scale'),
    'random_forest': RandomForestClassifier(n_estimators=200, random_state=42),
    'pls_da': PLSDAClassifier(n_components=2),
}

if TENSORFLOW_AVAILABLE:
    modelos['cnn_1d'] = CNN1DClassifier(epochs=30, batch_size=16, validation_split=0.2, verbose=0)
else:
    print('TensorFlow não encontrado: o modelo 1D-CNN será listado, mas não poderá ser treinado até instalar tensorflow.')
    modelos['cnn_1d'] = CNN1DClassifier(epochs=30, batch_size=16, validation_split=0.2, verbose=0)


# =========================================================
# K-FOLD CROSS VALIDATION
# =========================================================

kfold = StratifiedKFold(
    n_splits=8,
    shuffle=True,
    random_state=42,
)


# =========================================================
# PASTAS DE SAÍDA
# =========================================================

os.makedirs(f'models_intra_classe/laser{laser}', exist_ok=True)
os.makedirs(f'plots_intra_classe/laser{laser}', exist_ok=True)


# =========================================================
# TREINAMENTO E AVALIAÇÃO
# =========================================================

results = {}
print('Executando Cross Validation...')

for prep_name, prep_steps in preprocessamentos.items():
    for model_name, model in modelos.items():
        pipeline_name = f'{prep_name}_{model_name}'
        print(f'Pipeline: {pipeline_name}')

        pipe = Pipeline([
            *prep_steps,
            ('model', model),
        ])

        try:
            scores = cross_val_score(
                pipe,
                X,
                y,
                cv=kfold,
                scoring='accuracy',
            )
        except Exception as exc:
            print(f'Falha ao validar {pipeline_name}: {exc}')
            continue

        mean_acc = scores.mean()
        std_acc = scores.std()
        results[pipeline_name] = mean_acc

        print(f'Acurácias: {scores}')
        print(f'Média: {mean_acc:.4f}')
        print(f'Desvio padrão: {std_acc:.4f}')

        try:
            pipe.fit(X, y)

            if model_name == 'cnn_1d' and TENSORFLOW_AVAILABLE:
                model_path = f'models/laser{laser}/{pipeline_name}.keras'
                pipe.named_steps['model'].model_.save(model_path)
            else:
                model_path = f'models/laser{laser}/{pipeline_name}.joblib'
                joblib.dump(pipe, model_path)

            print(f'Modelo salvo em: {model_path}')
        except Exception as exc:
            print(f'Não foi possível salvar {pipeline_name}: {exc}')

        try:
            y_pred = cross_val_predict(pipe, X, y, cv=kfold)
            cm = confusion_matrix(y, y_pred)
            disp = ConfusionMatrixDisplay(
                confusion_matrix=cm,
                display_labels=['Arabica', 'Robusta'],
            )

            fig, ax = plt.subplots(figsize=(5, 5))
            disp.plot(ax=ax)
            plt.title(f'Matriz de Confusão - {pipeline_name}')

            plot_path = f'plots/laser{laser}/{pipeline_name}_cm.png'
            plt.savefig(plot_path, dpi=300, bbox_inches='tight')
            plt.close()
            print(f'Plot salvo em: {plot_path}')
        except Exception as exc:
            print(f'Não foi possível gerar matriz de confusão para {pipeline_name}: {exc}')

        summary_path = f'plots/laser{laser}/{pipeline_name}_summary.json'
        save_pipeline_summary_json(summary_path, pipeline_name, scores, mean_acc, std_acc, laser, np.unique(y), kfold.get_n_splits())
        print(f'Resumo salvo em: {summary_path}')


# =========================================================
# RANKING FINAL
# =========================================================

print('==============================')
print('RANKING FINAL')
print('==============================')

for k, v in sorted(results.items(), key=lambda item: item[1], reverse=True):
    print(f'{k}: {v:.4f}')


# =========================================================
# PLOT FINAL
# =========================================================

if results:
    names = list(results.keys())
    values = list(results.values())

    plt.figure(figsize=(14, 6))
    bars = plt.bar(names, values)
    plt.ylabel('Accuracy')
    plt.xlabel('Pipeline')
    plt.title(f'Comparação dos Pipelines - Laser {laser}')
    plt.ylim(0, 1)
    plt.xticks(rotation=45, ha='right')

    for bar in bars:
        height = bar.get_height()
        plt.text(
            bar.get_x() + bar.get_width() / 2,
            height + 0.01,
            f'{height:.3f}',
            ha='center',
        )

    plt.tight_layout()
    final_plot = f'plots/laser{laser}/ranking_final.png'
    plt.savefig(final_plot, dpi=300)
    plt.close()
    print(f'Plot final salvo em: {final_plot}')

print('Processo concluído!')


Lendo dataset...


C:\Users\Pedro\AppData\Local\Temp\ipykernel_18544\570081096.py:204: DtypeWarning: Columns (0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20,21,22,23,24,25,26,27,28,29,30,31,32,33,34,35,36,37,38,39,40,41,42,43,44,45,46,47,48,49,50,51,52,53,54,55,56,57,58,59,60,61,62,63,64,65,66,67,68,69,70,71,72,73,74,75,76,77,78,79,80,81,82,83,84,85,86,87,88,89,90,91,92,93,94,95,96,97,98,99,100,101,102,103,104,105,106,107,108,109,110,111,112,113,114,115,116,117,118,119,120,121,122,123,124,125,126,127,128,129,130,131,132,133,134,135,136,137,138,139,140,141,142,143,144,145,146,147,148,149,150,151,152,153,154,155,156,157,158,159,160,161,162,163,164,165,166,167,168,169,170,171,172,173,174,175,176,177,178,179,180,181,182,183,184,185,186,187,188,189,190,191,192,193,194,195,196,197,198,199,200,201,202,203,204,205,206,207,208,209,210,211,212,213,214,215,216,217,218,219,220,221,222,223,224,225,226,227,228,229,230,231,232,233,234,235,236,237,238,239,240,241,242,243,244,245,246,247,248,249,250,251,252,253,254

Formato do dataset: (272, 30262)
Labels: ['A' 'R']
Executando Cross Validation...
Pipeline: none_svm_linear
Acurácias: [1.         1.         1.         0.97058824 1.         0.97058824
 0.97058824 1.        ]
Média: 0.9890
Desvio padrão: 0.0142
Modelo salvo em: models/laser532/none_svm_linear.joblib
Plot salvo em: plots/laser532/none_svm_linear_cm.png
Resumo salvo em: plots/laser532/none_svm_linear_summary.json
Pipeline: none_svm_rbf
Acurácias: [0.94117647 1.         0.91176471 0.82352941 0.82352941 0.94117647
 0.91176471 0.88235294]
Média: 0.9044
Desvio padrão: 0.0565
Modelo salvo em: models/laser532/none_svm_rbf.joblib
Plot salvo em: plots/laser532/none_svm_rbf_cm.png
Resumo salvo em: plots/laser532/none_svm_rbf_summary.json
Pipeline: none_svm_poly
Acurácias: [0.76470588 0.97058824 0.82352941 0.70588235 0.82352941 0.70588235
 0.82352941 0.67647059]
Média: 0.7868
Desvio padrão: 0.0891
Modelo salvo em: models/laser532/none_svm_poly.joblib
Plot salvo em: plots/laser532/none_svm_poly_cm

c:\Users\Pedro\miniconda3\envs\LIBS\lib\site-packages\sklearn\model_selection\_validation.py:516: FitFailedWarning: 
3 fits failed out of a total of 8.
The score on these train-test partitions for these parameters will be set to nan.
If these failures are not expected, you can try to debug them by setting error_score='raise'.

Below are more details about the failures:
--------------------------------------------------------------------------------
1 fits failed with the following error:
Traceback (most recent call last):
  File "c:\Users\Pedro\miniconda3\envs\LIBS\lib\site-packages\sklearn\model_selection\_validation.py", line 859, in _fit_and_score
    estimator.fit(X_train, y_train, **fit_params)
  File "c:\Users\Pedro\miniconda3\envs\LIBS\lib\site-packages\sklearn\base.py", line 1365, in wrapper
    return fit_method(estimator, *args, **kwargs)
  File "c:\Users\Pedro\miniconda3\envs\LIBS\lib\site-packages\sklearn\pipeline.py", line 663, in fit
    self._final_estimator.fit(Xt, y, *

Acurácias: [0.91176471 0.88235294 0.97058824 0.85294118 0.97058824        nan
        nan        nan]
Média: nan
Desvio padrão: nan
Não foi possível salvar savgol_snv_scaler_cnn_1d: Unable to allocate 62.8 MiB for an array with shape (272, 30262) and data type float64
Não foi possível gerar matriz de confusão para savgol_snv_scaler_cnn_1d: Unable to allocate 54.9 MiB for an array with shape (238, 30262) and data type float64
Resumo salvo em: plots/laser532/savgol_snv_scaler_cnn_1d_summary.json
RANKING FINAL
savgol_snv_svm_linear: 1.0000
savgol_svm_linear: 0.9963
snv_svm_linear: 0.9963
none_svm_linear: 0.9890
scaler_svm_linear: 0.9890
savgol_snv_random_forest: 0.9853
savgol_snv_scaler_random_forest: 0.9853
savgol_snv_scaler_svm_linear: 0.9779
none_random_forest: 0.9743
scaler_random_forest: 0.9743
savgol_snv_scaler_svm_rbf: 0.9743
snv_random_forest: 0.9743
savgol_random_forest: 0.9706
scaler_svm_rbf: 0.9669
savgol_snv_scaler_svm_sigmoid: 0.9596
snv_pls_da: 0.9412
savgol_snv_pls_da: 0.92

pipeline com todos os modelos fazendo classificacao intra-classe com k-fold

In [6]:
# Código de classificação multi-classe (intra-classe) com K-Fold
import os
import re
import joblib
import json
from collections import Counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from scipy.signal import savgol_filter

from sklearn.base import BaseEstimator, ClassifierMixin, TransformerMixin
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import StratifiedKFold, cross_val_predict, cross_val_score
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC

# Reuso de transformadores definidos acima (SavgolFilter, SNV)

class CNN1DClassifier(BaseEstimator, ClassifierMixin):
    """Wrapper scikit-learn para 1D-CNN multi-classe com TensorFlow/Keras."""

    def __init__(self, epochs=30, batch_size=16, validation_split=0.2, verbose=0):
        self.epochs = epochs
        self.batch_size = batch_size
        self.validation_split = validation_split
        self.verbose = verbose

    def _build_model(self, input_length, n_classes):
        model = Sequential([
            InputLayer(shape=(input_length, 1)),
            Conv1D(16, kernel_size=5, activation='relu', padding='same'),
            MaxPooling1D(pool_size=2),
            Conv1D(32, kernel_size=3, activation='relu', padding='same'),
            MaxPooling1D(pool_size=2),
            Flatten(),
            Dense(64, activation='relu'),
            Dropout(0.3),
            Dense(n_classes, activation='softmax'),
        ])
        model.compile(
            optimizer='adam',
            loss='sparse_categorical_crossentropy',
            metrics=['accuracy'],
        )
        return model

    def fit(self, X, y):
        if not TENSORFLOW_AVAILABLE:
            raise ImportError('TensorFlow não está instalado. Instale tensorflow para usar o modelo 1D-CNN.')

        self.label_encoder_ = LabelEncoder()
        y_encoded = self.label_encoder_.fit_transform(y).astype(np.int32)
        self.n_classes_ = len(self.label_encoder_.classes_)
        if self.n_classes_ < 2:
            raise ValueError('O modelo 1D-CNN nesta implementação espera pelo menos duas classes.')

        X_cnn = X.astype(np.float32)[..., np.newaxis]
        self.model_ = self._build_model(X_cnn.shape[1], self.n_classes_)
        callback = EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)
        self.model_.fit(
            X_cnn,
            y_encoded,
            epochs=self.epochs,
            batch_size=self.batch_size,
            validation_split=self.validation_split,
            verbose=self.verbose,
            callbacks=[callback],
        )
        return self

    def predict(self, X):
        X_cnn = X.astype(np.float32)[..., np.newaxis]
        probabilities = self.model_.predict(X_cnn, verbose=0)
        predicted = np.argmax(probabilities, axis=1)
        return self.label_encoder_.inverse_transform(predicted)

print('\n--- Iniciando pipeline multi-classe ---')
laser = int(input('Qual laser utilizado? 532 ou 1064\n'))
if laser not in [532, 1064]:
    raise ValueError('O valor deve ser 532 ou 1064')

print('Lendo dataset...')
file_path = f'dataset/laser{laser}.dat'
df = pd.read_csv(file_path, sep=';', header=None)

y_raw = df.iloc[0, 1:].values.astype(str)
X_full = df.iloc[1:, 1:].values.astype(float).T

def extrair_numero(label):
    match = re.search(r'\d+', label)
    return int(match.group()) if match else None

# Separar cada rótulo (ex: A1, A2, R1, ...) respeitando os limites do filtro original
X = []
y = []
for xi, yi in zip(X_full, y_raw):
    num = extrair_numero(yi)
    if laser == 532:
        # manter apenas rótulos com número <= 8 (mesma regra do script binário)
        if (yi.startswith('A') or yi.startswith('R')) and num is not None and num <= 8:
            X.append(xi)
            y.append(yi)
    else:  # laser == 1064
        # regras do script binário: A <= 12, R <= 9
        if yi.startswith('A') and num is not None and num <= 12:
            X.append(xi)
            y.append(yi)
        elif yi.startswith('R') and num is not None and num <= 9:
            X.append(xi)
            y.append(yi)

X = np.array(X)
y = np.array(y)

print(f'Formato do dataset (X): {X.shape}')
unique_labels = np.unique(y)
print(f'Número de classes: {len(unique_labels)}')
print(f'Rótulos: {unique_labels}')

# Se alguma classe tiver menos de 2 amostras, StratifiedKFold falhará. Avisar/ajustar automaticamente
class_counts = Counter(y)
min_count = min(class_counts.values()) if class_counts else 0
if min_count < 2:
    raise ValueError('Algumas classes têm menos de 2 amostras. Ajuste os filtros ou aumente o número de amostras por classe.')

n_splits = min(8, min_count)
kfold = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)

# Preprocessamentos e modelos (similar ao script binário, mas mantendo modelos compatíveis com multi-classe)
def make_savgol(window_length=11, polyorder=2):
    return ('savgol', SavgolFilter(window_length=window_length, polyorder=polyorder))
def make_snv():
    return ('snv', SNV())
def make_scaler():
    return ('scaler', StandardScaler())


def save_pipeline_summary_json(output_path, payload):
    with open(output_path, 'w', encoding='utf-8') as json_file:
        json.dump(payload, json_file, ensure_ascii=False, indent=2)

preprocessamentos = {
    'none': [],
    'snv': [make_snv()],
    'savgol_snv': [make_savgol(), make_snv()],
}

modelos = {
    'svm_linear': SVC(kernel='linear', gamma='scale'),
    'random_forest': RandomForestClassifier(n_estimators=200, random_state=42),
    'pls_da': PLSDAMulticlass(n_components=10),
}

if TENSORFLOW_AVAILABLE:
    #modelos['cnn_1d'] = CNN1DClassifier(epochs=30, batch_size=16, validation_split=0.2, verbose=0)
    print("por enquanto deixado de fora")
else:
    print('TensorFlow não encontrado: o modelo 1D-CNN multi-classe não poderá ser treinado até instalar tensorflow.')

os.makedirs(f'models_intra_classe/laser{laser}', exist_ok=True)
os.makedirs(f'plots_intra_classe/laser{laser}', exist_ok=True)

results = {}
pipeline_summaries = []
print('Executando Cross Validation multi-classe...')

for prep_name, prep_steps in preprocessamentos.items():
    for model_name, model in modelos.items():
        pipeline_name = f'{prep_name}_{model_name}'
        print(f'Pipeline: {pipeline_name}')

        pipe = Pipeline([
            *prep_steps,
            ('model', model),
        ])

        try:
            scores = cross_val_score(pipe, X, y, cv=kfold, scoring='accuracy')
        except Exception as exc:
            print(f'Falha ao validar {pipeline_name}: {exc}')
            continue

        mean_acc = scores.mean()
        std_acc = scores.std()
        results[pipeline_name] = mean_acc

        print(f'Acurácias: {scores}')
        print(f'Média: {mean_acc:.4f}')
        print(f'Desvio padrão: {std_acc:.4f}')

        try:
            pipe.fit(X, y)
            if model_name == 'cnn_1d' and TENSORFLOW_AVAILABLE:
                model_path = f'models_intra_classe/laser{laser}/{pipeline_name}.keras'
                pipe.named_steps['model'].model_.save(model_path)
            else:
                model_path = f'models_intra_classe/laser{laser}/{pipeline_name}.joblib'
                joblib.dump(pipe, model_path)
            print(f'Modelo salvo em: {model_path}')
        except Exception as exc:
            print(f'Não foi possível salvar {pipeline_name}: {exc}')

        try:
            y_pred = cross_val_predict(pipe, X, y, cv=kfold)
            cm = confusion_matrix(y, y_pred, labels=sorted(unique_labels))
            disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=sorted(unique_labels))

            fig, ax = plt.subplots(figsize=(6, 6))
            disp.plot(ax=ax, xticks_rotation='vertical')
            plt.title(f'Matriz de Confusão - {pipeline_name}')

            plot_path = f'plots_intra_classe/laser{laser}/{pipeline_name}_cm.png'
            plt.savefig(plot_path, dpi=300, bbox_inches='tight')
            plt.close()
            print(f'Plot salvo em: {plot_path}')
        except Exception as exc:
            print(f'Não foi possível gerar matriz de confusão para {pipeline_name}: {exc}')

        pipeline_summaries.append({
            'pipeline': pipeline_name,
            'labels': [str(label) for label in unique_labels],
            'n_splits': kfold.get_n_splits(),
            'accuracy_scores': [float(score) for score in scores],
            'mean_accuracy': float(mean_acc),
            'std_accuracy': float(std_acc),
        })

consolidated_summary_path = f'plots_intra_classe/laser{laser}/all_pipelines_summary.json'
save_pipeline_summary_json(consolidated_summary_path, {
    'experiment': 'intra_classe_kfold',
    'laser': laser,
    'labels': [str(label) for label in unique_labels],
    'n_splits': kfold.get_n_splits(),
    'pipelines': pipeline_summaries,
})
print(f'Resumo consolidado salvo em: {consolidated_summary_path}')

# Ranking final
print('==============================')
print('RANKING FINAL (multi-classe)')
print('==============================')
for k, v in sorted(results.items(), key=lambda item: item[1], reverse=True):
    print(f'{k}: {v:.4f}')

# Plot final
if results:
    names = list(results.keys())
    values = list(results.values())

    plt.figure(figsize=(14, 6))
    bars = plt.bar(names, values)
    plt.ylabel('Accuracy')
    plt.xlabel('Pipeline')
    plt.title(f'Comparação dos Pipelines - Laser {laser} (multi-classe)')
    plt.ylim(0, 1)
    plt.xticks(rotation=45, ha='right')

    for bar in bars:
        height = bar.get_height()
        plt.text(bar.get_x() + bar.get_width() / 2, height + 0.01, f'{height:.3f}', ha='center')

    plt.tight_layout()
    final_plot = f'plots_intra_classe/laser{laser}/ranking_final_multiclass.png'
    plt.savefig(final_plot, dpi=300)
    plt.close()
    print(f'Plot final salvo em: {final_plot}')

print('Processo multi-classe concluído!')


--- Iniciando pipeline multi-classe ---
Lendo dataset...


C:\Users\Pedro\AppData\Local\Temp\ipykernel_1092\2488085329.py:89: DtypeWarning: Columns (0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20,21,22,23,24,25,26,27,28,29,30,31,32,33,34,35,36,37,38,39,40,41,42,43,44,45,46,47,48,49,50,51,52,53,54,55,56,57,58,59,60,61,62,63,64,65,66,67,68,69,70,71,72,73,74,75,76,77,78,79,80,81,82,83,84,85,86,87,88,89,90,91,92,93,94,95,96,97,98,99,100,101,102,103,104,105,106,107,108,109,110,111,112,113,114,115,116,117,118,119,120,121,122,123,124,125,126,127,128,129,130,131,132,133,134,135,136,137,138,139,140,141,142,143,144,145,146,147,148,149,150,151,152,153,154,155,156,157,158,159,160,161,162,163,164,165,166,167,168,169,170,171,172,173,174,175,176,177,178,179,180,181,182,183,184,185,186,187,188,189,190,191,192,193,194,195,196,197,198,199,200,201,202,203,204,205,206,207,208,209,210,211,212,213,214,215,216,217,218,219,220,221,222,223,224,225,226,227,228,229,230,231,232,233,234,235,236,237,238,239,240,241,242,243,244,245,246,247,248,249,250,251,252,253,254,

Formato do dataset (X): (272, 30262)
Número de classes: 17
Rótulos: ['A1' 'A2' 'A2+CA2' 'A3' 'A4' 'A5' 'A6' 'A7' 'A8' 'R1' 'R2' 'R3' 'R4' 'R5'
 'R6' 'R7' 'R8']
por enquanto deixado de fora
Executando Cross Validation multi-classe...
Pipeline: none_svm_linear
Acurácias: [0.73529412 0.88235294 0.61764706 0.76470588 0.82352941 0.85294118
 0.82352941 0.61764706]
Média: 0.7647
Desvio padrão: 0.0953
Modelo salvo em: models_intra_classe/laser532/none_svm_linear.joblib
Plot salvo em: plots_intra_classe/laser532/none_svm_linear_cm.png
Pipeline: none_random_forest
Acurácias: [0.67647059 0.70588235 0.73529412 0.58823529 0.76470588 0.79411765
 0.61764706 0.67647059]
Média: 0.6949
Desvio padrão: 0.0657
Modelo salvo em: models_intra_classe/laser532/none_random_forest.joblib
Plot salvo em: plots_intra_classe/laser532/none_random_forest_cm.png
Pipeline: none_pls_da
Acurácias: [0.64705882 0.73529412 0.58823529 0.58823529 0.58823529 0.70588235
 0.61764706 0.64705882]
Média: 0.6397
Desvio padrão: 0.0525


pipeline com todos os modelos fazendo classificacao binaria com group-k-fold

In [ ]:
import os
import re
import joblib
import json

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import GroupKFold, cross_val_predict, cross_val_score
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

# =========================================================
# ESCOLHA DO LASER
# =========================================================

laser = int(input('Qual laser utilizado? 532 ou 1064\n'))

if laser not in [532, 1064]:
    raise ValueError('O valor deve ser 532 ou 1064')


# =========================================================
# LEITURA DO DATASET
# =========================================================

print('\nLendo dataset...')
file_path = f'dataset/laser{laser}.dat'

df = pd.read_csv(file_path, sep=';', header=None)

y_raw = df.iloc[0, 1:].values.astype(str)
X_full = df.iloc[1:, 1:].values.astype(float).T


# =========================================================
# SEPARAÇÃO DAS CLASSES
# =========================================================

def extrair_numero(label):
    match = re.search(r'\d+', label)
    return int(match.group()) if match else None


X = []
y = []
groups = []

for xi, yi in zip(X_full, y_raw):
    num = extrair_numero(yi)
    if laser == 532:
        if yi.startswith('A') and num is not None and num <= 8:
            X.append(xi)
            y.append('A')
            groups.append(yi)
        elif yi.startswith('R') and num is not None and num <= 8:
            X.append(xi)
            y.append('R')
            groups.append(yi)
    else:
        if yi.startswith('A') and num is not None and num <= 12:
            X.append(xi)
            y.append('A')
            groups.append(yi)
        elif yi.startswith('R') and num is not None and num <= 9:
            X.append(xi)
            y.append('R')
            groups.append(yi)

X = np.array(X)
y = np.array(y)
groups = np.array(groups)


def save_pipeline_summary_json(output_path, pipeline_name, scores, mean_acc, std_acc, laser, labels, n_splits):
    summary = {
        'pipeline': pipeline_name,
        'laser': laser,
        'labels': [str(label) for label in labels],
        'n_splits': n_splits,
        'accuracy_scores': [float(score) for score in scores],
        'mean_accuracy': float(mean_acc),
        'std_accuracy': float(std_acc),
    }
    with open(output_path, 'w', encoding='utf-8') as json_file:
        json.dump(summary, json_file, ensure_ascii=False, indent=2)

print(f'Formato do dataset: {X.shape}')
print(f'Labels: {np.unique(y)}')
print(f'Grupos: {np.unique(groups)}')

unique_groups = np.unique(groups)
if len(unique_groups) < 2:
    raise ValueError('É necessário pelo menos 2 grupos para usar GroupKFold.')

n_splits = min(8, len(unique_groups))
gkf = GroupKFold(n_splits=n_splits)


# =========================================================
# CONFIGURAÇÃO DOS EXPERIMENTOS
# =========================================================


preprocessamentos = {
    'none': [],
    'snv': [make_snv()],
    'savgol_snv': [make_savgol(), make_snv()],
}

modelos = {
    'svm_linear': SVC(kernel='linear', gamma='scale'),
    'random_forest': RandomForestClassifier(n_estimators=200, random_state=42),
    'pls_da': PLSDAClassifier(n_components=2),
}

if TENSORFLOW_AVAILABLE:
    #modelos['cnn_1d'] = CNN1DClassifier(epochs=30, batch_size=16, validation_split=0.2, verbose=0)
    print("por enquanto deixado de fora")
else:
    print('TensorFlow não encontrado: o modelo 1D-CNN será listado, mas não poderá ser treinado até instalar tensorflow.')
    modelos['cnn_1d'] = CNN1DClassifier(epochs=30, batch_size=16, validation_split=0.2, verbose=0)


# =========================================================
# PASTAS DE SAÍDA
# =========================================================

os.makedirs(f'models_groupkfold/laser{laser}', exist_ok=True)
os.makedirs(f'plots_groupkfold/laser{laser}', exist_ok=True)


# =========================================================
# TREINAMENTO E AVALIAÇÃO
# =========================================================

results = {}
pipeline_summaries = []
print('Executando Cross Validation com GroupKFold...')

for prep_name, prep_steps in preprocessamentos.items():
    for model_name, model in modelos.items():
        pipeline_name = f'{prep_name}_{model_name}'
        print(f'Pipeline: {pipeline_name}')

        pipe = Pipeline([
            *prep_steps,
            ('model', model),
        ])

        try:
            scores = cross_val_score(
                pipe,
                X,
                y,
                cv=gkf,
                groups=groups,
                scoring='accuracy',
            )
        except Exception as exc:
            print(f'Falha ao validar {pipeline_name}: {exc}')
            continue

        mean_acc = scores.mean()
        std_acc = scores.std()
        results[pipeline_name] = mean_acc

        print(f'Acurácias: {scores}')
        print(f'Média: {mean_acc:.4f}')
        print(f'Desvio padrão: {std_acc:.4f}')

        try:
            pipe.fit(X, y)

            if model_name == 'cnn_1d' and TENSORFLOW_AVAILABLE:
                model_path = f'models_groupkfold/laser{laser}/{pipeline_name}.keras'
                pipe.named_steps['model'].model_.save(model_path)
            else:
                model_path = f'models_groupkfold/laser{laser}/{pipeline_name}.joblib'
                joblib.dump(pipe, model_path)

            print(f'Modelo salvo em: {model_path}')
        except Exception as exc:
            print(f'Não foi possível salvar {pipeline_name}: {exc}')

        try:
            y_pred = cross_val_predict(
                pipe,
                X,
                y,
                cv=gkf,
                groups=groups,
            )
            cm = confusion_matrix(y, y_pred)
            disp = ConfusionMatrixDisplay(
                confusion_matrix=cm,
                display_labels=['Arabica', 'Robusta'],
            )

            fig, ax = plt.subplots(figsize=(5, 5))
            disp.plot(ax=ax)
            plt.title(f'Matriz de Confusão - {pipeline_name}')

            plot_path = f'plots_groupkfold/laser{laser}/{pipeline_name}_cm.png'
            plt.savefig(plot_path, dpi=300, bbox_inches='tight')
            plt.close()
            print(f'Plot salvo em: {plot_path}')
        except Exception as exc:
            print(f'Não foi possível gerar matriz de confusão para {pipeline_name}: {exc}')

        pipeline_summaries.append({
            'pipeline': pipeline_name,
            'labels': [str(label) for label in np.unique(y)],
            'n_splits': gkf.get_n_splits(),
            'accuracy_scores': [float(score) for score in scores],
            'mean_accuracy': float(mean_acc),
            'std_accuracy': float(std_acc),
        })

consolidated_summary_path = f'plots_groupkfold/laser{laser}/all_pipelines_summary.json'

summary = {
    'experiment': 'groupkfold_binary',
    'laser': laser,
    'labels': [str(label) for label in np.unique(y)],
    'n_splits': gkf.get_n_splits(),
    'pipelines': pipeline_summaries,
}

with open(consolidated_summary_path, 'w', encoding='utf-8') as json_file:
    json.dump(summary, json_file, ensure_ascii=False, indent=2)

print(f'Resumo consolidado salvo em: {consolidated_summary_path}')


# =========================================================
# RANKING FINAL
# =========================================================

print('==============================')
print('RANKING FINAL')
print('==============================')

for k, v in sorted(results.items(), key=lambda item: item[1], reverse=True):
    print(f'{k}: {v:.4f}')


# =========================================================
# PLOT FINAL
# =========================================================

if results:
    names = list(results.keys())
    values = list(results.values())

    plt.figure(figsize=(14, 6))
    bars = plt.bar(names, values)
    plt.ylabel('Accuracy')
    plt.xlabel('Pipeline')
    plt.title(f'Comparação dos Pipelines - Laser {laser} (GroupKFold)')
    plt.ylim(0, 1)
    plt.xticks(rotation=45, ha='right')

    for bar in bars:
        height = bar.get_height()
        plt.text(
            bar.get_x() + bar.get_width() / 2,
            height + 0.01,
            f'{height:.3f}',
            ha='center',
        )

    plt.tight_layout()
    final_plot = f'plots_groupkfold/laser{laser}/ranking_final_groupkfold.png'
    plt.savefig(final_plot, dpi=300)
    plt.close()
    print(f'Plot final salvo em: {final_plot}')

print('Processo concluído!')


Lendo dataset...


C:\Users\Pedro\AppData\Local\Temp\ipykernel_1092\2253884836.py:30: DtypeWarning: Columns (0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20,21,22,23,24,25,26,27,28,29,30,31,32,33,34,35,36,37,38,39,40,41,42,43,44,45,46,47,48,49,50,51,52,53,54,55,56,57,58,59,60,61,62,63,64,65,66,67,68,69,70,71,72,73,74,75,76,77,78,79,80,81,82,83,84,85,86,87,88,89,90,91,92,93,94,95,96,97,98,99,100,101,102,103,104,105,106,107,108,109,110,111,112,113,114,115,116,117,118,119,120,121,122,123,124,125,126,127,128,129,130,131,132,133,134,135,136,137,138,139,140,141,142,143,144,145,146,147,148,149,150,151,152,153,154,155,156,157,158,159,160,161,162,163,164,165,166,167,168,169,170,171,172,173,174,175,176,177,178,179,180,181,182,183,184,185,186,187,188,189,190,191,192,193,194,195,196,197,198,199,200,201,202,203,204,205,206,207,208,209,210,211,212,213,214,215,216,217,218,219,220,221,222,223,224,225,226,227,228,229,230,231,232,233,234,235,236,237,238,239,240,241,242,243,244,245,246,247,248,249,250,251,252,253,254,

Formato do dataset: (272, 30262)
Labels: ['A' 'R']
Grupos: [1 2 3 4 5 6 7 8]
por enquanto deixado de fora
Executando Cross Validation com GroupKFold...
Pipeline: none_svm_linear
Acurácias: [0.625   1.      0.90625 0.96875 1.      0.75    1.      0.5625 ]
Média: 0.8516
Desvio padrão: 0.1688
Modelo salvo em: models_groupkfold/laser532/none_svm_linear.joblib
Plot salvo em: plots_groupkfold/laser532/none_svm_linear_cm.png
Pipeline: none_random_forest
Acurácias: [0.95833333 1.         0.96875    0.9375     0.96875    0.84375
 1.         0.75      ]
Média: 0.9284
Desvio padrão: 0.0818
Modelo salvo em: models_groupkfold/laser532/none_random_forest.joblib
Plot salvo em: plots_groupkfold/laser532/none_random_forest_cm.png
Pipeline: none_pls_da
Acurácias: [0.875   0.90625 0.84375 0.90625 0.625   0.5     1.      0.46875]
Média: 0.7656
Desvio padrão: 0.1907
Modelo salvo em: models_groupkfold/laser532/none_pls_da.joblib
Plot salvo em: plots_groupkfold/laser532/none_pls_da_cm.png
Pipeline: snv_svm_l

In [5]:
import json
from pathlib import Path

base_dir = Path(r'c:\Users\Pedro\OneDrive\Documentos\LIBS_NEW')
output_path = base_dir / 'plots' / 'consolidated_all_pipelines.json'

sources = [
    ('binary_kfold', base_dir / 'plots'),
    ('intra_classe_kfold', base_dir / 'plots_intra_classe'),
    ('groupkfold_binary', base_dir / 'plots_groupkfold'),
]

def normalize_missing_values(value):
    # Usa -1 no lugar de NaN/None para manter o JSON consistente em qualquer leitor.
    if isinstance(value, dict):
        return {key: normalize_missing_values(item) for key, item in value.items()}
    if isinstance(value, list):
        return [normalize_missing_values(item) for item in value]
    if value is None:
        return -1
    if isinstance(value, float) and value != value:
        return -1
    return value

consolidated = {'lasers': {}}

for experiment_name, root_folder in sources:
    if not root_folder.exists():
        continue

    for laser in [532, 1064]:
        laser_folder = root_folder / f'laser{laser}'
        if not laser_folder.exists():
            continue

        summary_file = laser_folder / 'all_pipelines_summary.json'
        if summary_file.exists():
            with open(summary_file, 'r', encoding='utf-8') as file_handle:
                summary_data = json.load(file_handle)
            summary_data = normalize_missing_values(summary_data)
        else:
            pipeline_files = sorted(
                path for path in laser_folder.glob('*_summary.json')
                if path.name != 'all_pipelines_summary.json'
            )
            pipelines = []
            for pipeline_file in pipeline_files:
                with open(pipeline_file, 'r', encoding='utf-8') as file_handle:
                    pipeline_data = json.load(file_handle)
                pipelines.append(normalize_missing_values(pipeline_data))
            summary_data = {
                'experiment': experiment_name,
                'laser': laser,
                'labels': [],
                'n_splits': -1,  # Usa -1 no lugar de NaN/None para manter o JSON válido em qualquer parser.
                'pipelines': pipelines,
            }

        summary_data['experiment'] = experiment_name
        consolidated['lasers'].setdefault(str(laser), {})[experiment_name] = summary_data

output_path.parent.mkdir(parents=True, exist_ok=True)
with open(output_path, 'w', encoding='utf-8') as file_handle:
    json.dump(consolidated, file_handle, ensure_ascii=False, indent=2)

print(f'Arquivo consolidado salvo em: {output_path}')

Arquivo consolidado salvo em: c:\Users\Pedro\OneDrive\Documentos\LIBS_NEW\plots\consolidated_all_pipelines.json
